In [1]:
import pandas as pd
import glob
import os

In [2]:
path  = './busan_trip_jsons'
all_files = glob.glob(os.path.join(path, "*.json"))

In [3]:
df_list = [pd.read_json(f) for f in all_files]

df = pd.concat(df_list, ignore_index = True)

In [4]:
# 1. 공백이나 빈 문자열을 실제 NaN으로 치환하여 개수 확인
# strip()으로 양끝 공백을 지운 뒤 빈 값인 것들을 체크합니다.
missing_summary = df.apply(lambda x: x.astype(str).str.strip().replace(['', 'None', 'nan', '-'], pd.NA)).isnull().sum()

print("--- 실제 비어있는 데이터(공백/특수문자 포함) ---")
print(missing_summary)

--- 실제 비어있는 데이터(공백/특수문자 포함) ---
장소명       0
구분        0
카테고리    333
설명        0
메뉴      351
영업시간    334
주소       38
연락처     187
출처        0
dtype: int64


In [5]:
df['구분'].value_counts()

구분
음식점     530
관광지     169
카페       94
체험       90
문화예술     44
쇼핑       32
숙박업체     18
축제        4
사찰        2
Name: count, dtype: int64

In [6]:
df['구분'] = df['구분'].replace('사찰', '관광지')

# 결과 확인 (제대로 바뀌었는지 개수 체크)
print(df['구분'].value_counts())

구분
음식점     530
관광지     171
카페       94
체험       90
문화예술     44
쇼핑       32
숙박업체     18
축제        4
Name: count, dtype: int64


## 공백과 줄바꿈을 제거

In [7]:
df['장소명_공백제거'] = df['장소명'].str.replace(r'\s+', '', regex=True)
df['설명_공백제거'] = df['설명'].str.replace(r'\s+', '', regex=True)


## 장소명은 같은데 구분이 다른 데이터들 처리

In [8]:
# 1. 장소명별로 '구분'의 유니크한 개수가 2개 이상인 장소들 찾기
diff_category_names = df.groupby('장소명_공백제거')['구분'].nunique()
diff_category_names = diff_category_names[diff_category_names > 1].index

# 2. 해당 장소들에 해당하는 전체 데이터 추출
df_diff_cat = df[df['장소명_공백제거'].isin(diff_category_names)]

# 3. 보기 좋게 정렬 (장소명 기준)
df_diff_cat_sorted = df_diff_cat.sort_values(by=['장소명_공백제거', '구분'])

# 4. 주요 컬럼 확인
print(f"📍 구분이 서로 다른 중복 장소 수: {len(diff_category_names)}곳")
df_diff_cat_sorted[['장소명', '구분', '출처', '설명']]

📍 구분이 서로 다른 중복 장소 수: 13곳


,장소명,구분,출처,설명
697,국립해양박물관,관광지,택슐랭(2024),"국립해양박물관은 “나의 바다, 우리의 미래”라는 컨셉으로 해양문화, 해양역사·인물,..."
391,국립해양박물관,문화예술,Theme(2025),해양의 모든 분야를 아우르는 종합해양박물관으로 바다의 도시 부산을 느낄 수 있다.
791,국립해양박물관,문화예술,베스트15(2025),"국립해양박물관은 바다의 문화, 역사, 과학 그리고 고고학에 이르는 전문적인 자료를 ..."
980,국립해양박물관,문화예술,부산한골목-봉산마을,우리는 대한민국 해양에 대해 얼마나 알고 있을까? 국립해양박물관은 부산 앞 바다를 ...
449,국제시장,관광지,Theme(2025),"먹자골목, 젊음의 거리, 만물의 거리, 깡통시장, 아리랑 거리, 구제 골목 등으로 ..."
777,국제시장,쇼핑,베스트15(2025),"먹자골목, 젊음의 거리, 만물의 거리, 깡통시장, 아리랑 거리, 구제 골목 등으로 ..."
383,더베이101,문화예술,Theme(2025),"해운대에 위치한 복합 문화 예술 공간. 바다와 산, 현대적인 건물들이 공존해 화려한..."
767,더베이101,체험,베스트15(2025),동백섬 일대의 핫플레이스로 자리 잡은 더베이101을 방문해보자. 요트 체험을 비롯한...
306,무슈뱅상,음식점,부산의 맛(2024),프랑스산 밀가루와 천연발효종인 르방을 사용하여 빵을 만드는 이곳은 20년간 빵을 만...
187,무슈뱅상,카페,블루리본(2023),"밀가루, 소금, 버터, 초콜릿 등 프랑스산 재료를 사용하는 프랑스식 베이커리 카페...."


## 설명이 0.8 이상인 데이터 확인

In [9]:
from difflib import SequenceMatcher

def similarity_ratio(a, b):
    return SequenceMatcher(None, str(a), str(b)).ratio()

# 유사한 데이터를 담을 리스트
similar_pairs = []

# 장소명이 같은 그룹 내에서만 비교 (효율성)
grouped = df.groupby('장소명_공백제거')

for name, group in grouped:
    if len(group) > 1:  # 동일 장소명이 2개 이상인 경우만
        contents = group['설명_공백제거'].tolist()
        indices = group.index.tolist()
        
        for i in range(len(contents)):
            for j in range(i + 1, len(contents)):
                score = similarity_ratio(contents[i], contents[j])
                if score >= 0.8:
                    similar_pairs.append({
                        '장소명': name,
                        '유사도': f"{score:.2%}",
                        '설명1': contents[i],
                        '설명2': contents[j],
                        '원본인덱스': (indices[i], indices[j])
                    })

# 결과 확인
df_similar = pd.DataFrame(similar_pairs)
print(f"80% 이상 유사한 세트 개수: {len(df_similar)}")

# % 기호를 빼고 숫자로 변환하여 정렬 (내림차순)
df_similar_sorted = df_similar.assign(
    sort_key = df_similar['유사도'].str.replace('%', '').astype(float)
).sort_values('sort_key', ascending=False).drop(columns='sort_key')

# 결과 확인
df_similar_sorted

80% 이상 유사한 세트 개수: 132


,장소명,유사도,설명1,설명2,원본인덱스
131,히떼로스터리전포점,100.00%,공간도커피도마지막까지편안하고부드러운것을지향하는이곳은밸런스좋고여운이깔끔한커피를완성하는...,공간도커피도마지막까지편안하고부드러운것을지향하는이곳은밸런스좋고여운이깔끔한커피를완성하는...,"(279, 549)"
0,60년전통할매국밥,100.00%,60년전통의돼지국밥전문점으로식사시간에는30분이상을기다려야하는유명한식당이다.대부분의돼...,60년전통의돼지국밥전문점으로식사시간에는30분이상을기다려야하는유명한식당이다.대부분의돼...,"(247, 516)"
1,611WoodFire,100.00%,오너셰프의스타일과부산의색을녹여낸우드파이어레스토랑은특색있지만편안한분위기를연출한다.나무...,오너셰프의스타일과부산의색을녹여낸우드파이어레스토랑은특색있지만편안한분위기를연출한다.나무...,"(296, 564)"
2,88돼지갈비,100.00%,숙련된솜씨로구워주는고기의맛을표현할만한방법이떠오르지않는다.입구부터투박하고오래된흔적을풍...,숙련된솜씨로구워주는고기의맛을표현할만한방법이떠오르지않는다.입구부터투박하고오래된흔적을풍...,"(248, 517)"
116,탐복,100.00%,"예쁜플레이팅에담긴국내산전복요리,시그니처인전복내장소스등전복요리를전문으로한캐주얼한식을선...","예쁜플레이팅에담긴국내산전복요리,시그니처인전복내장소스등전복요리를전문으로한캐주얼한식을선...","(238, 506)"
...,...,...,...,...,...
75,스카이라인루지부산,89.23%,2021년7월초개장한스카이라인루지부산.스카이라이드(리프트)를타고정상에위치한루지출발지...,2021년7월초개장한스카이라인루지부산은기장군오시리아관광단지내에자리했다.스카이라이드(...,"(431, 785)"
60,부산명물횟집,87.61%,"자갈치시장내에있는80여년전통의회백반집.푸짐하게나오는큼직큼직한생선살과초장,반찬이구미를...","80여년전통의회백반집.푸짐하게나오는큼직큼직한생선살과초장,반찬이구미를당긴다.회백반에는...","(171, 868)"
88,용두산공원,86.72%,120m높이에달하는부산타워전망대에서내려다보는부산의전경이특히인상적이다.도심속휴식공간으...,120m높이에달하는부산타워전망대에서내려다보는부산의모습이특히인상적이다.도심속휴식공간으...,"(371, 776)"
103,정통관,85.71%,매콤함과달콤함두마리토끼를잡은고추짜장과얼큰하고담백한차돌짬뽕을먹을수있는20년전통의중화요...,매콤함과달콤함두마리토끼를잡은고추짜장과얼큰하고담백한차돌짬뽕을먹을수있는20년전통의중화요...,"(647, 673)"


In [10]:
df_similar.iloc[0]['설명1']

'60년전통의돼지국밥전문점으로식사시간에는30분이상을기다려야하는유명한식당이다.대부분의돼지국밥들과는다른맑은국물이특징이며,대충썬것같은돼지수육은예상외로부드러운맛이다.돼지국물과고기가따로나오는수육백반이특히인기있다.'

In [11]:
df_similar.iloc[0]['설명2']

'60년전통의돼지국밥전문점으로식사시간에는30분이상을기다려야하는유명한식당이다.대부분의돼지국밥들과는다른맑은국물이특징이며,대충썬것같은돼지수육은예상외로부드러운맛이다.돼지국물과고기가따로나오는수육백반이특히인기있다.'

## 중복 제거

In [12]:
import re

# 1. 연도 포함 여부를 체크하는 함수 (True/False 반환)
def has_year(text):
    return bool(re.search(r'\d{4}', str(text)))

similar_check_list = []
grouped = df.groupby('장소명_공백제거')

for name, group in grouped:
    if len(group) > 1:
        contents = group['설명_공백제거'].tolist()
        sources = group['출처'].tolist()
        indices = group.index.tolist()
        
        for i in range(len(contents)):
            for j in range(i + 1, len(contents)):
                score = similarity_ratio(contents[i], contents[j])
                
                # 유사도가 0.8 이상인 경우만 기록
                if score >= 0.8:
                    similar_check_list.append({
                        '장소명': name,
                        '유사도': f"{score:.2%}",
                        '출처1': sources[i],
                        '연도1존재': has_year(sources[i]),
                        '출처2': sources[j],
                        '연도2존재': has_year(sources[j]),
                        '설명1(일부)': contents[i][:30],
                        '설명2(일부)': contents[j][:30]
                    })

# 데이터프레임으로 변환
df_check = pd.DataFrame(similar_check_list)

# 결과 요약 출력
if not df_check.empty:
    all_has_year = df_check['연도1존재'].all() and df_check['연도2존재'].all()
    print(f"✅ 유사 데이터 쌍 개수: {len(df_check)}")
    print(f"✅ 모든 데이터에 연도가 포함되어 있는가? : {all_has_year}")
    
    # 연도가 없는 데이터가 있다면 출력
    missing_year = df_check[(df_check['연도1존재'] == False) | (df_check['연도2존재'] == False)]
    if not missing_year.empty:
        print("\n⚠️ 연도 정보가 없는 데이터 요약:")
        print(missing_year[['장소명', '출처1', '연도1존재', '출처2', '연도2존재']])
else:
    print("유사도가 0.8 이상인 중복 데이터가 없습니다.")

df_check.head()

✅ 유사 데이터 쌍 개수: 132
✅ 모든 데이터에 연도가 포함되어 있는가? : True


,장소명,유사도,출처1,연도1존재,출처2,연도2존재,설명1(일부),설명2(일부)
0,60년전통할매국밥,100.00%,부산의 맛(2024),True,부산의 맛(2025),True,60년전통의돼지국밥전문점으로식사시간에는30분이상을기다려,60년전통의돼지국밥전문점으로식사시간에는30분이상을기다려
1,611WoodFire,100.00%,부산의 맛(2024),True,부산의 맛(2025),True,오너셰프의스타일과부산의색을녹여낸우드파이어레스토랑은특색있,오너셰프의스타일과부산의색을녹여낸우드파이어레스토랑은특색있
2,88돼지갈비,100.00%,부산의 맛(2024),True,부산의 맛(2025),True,숙련된솜씨로구워주는고기의맛을표현할만한방법이떠오르지않는다,숙련된솜씨로구워주는고기의맛을표현할만한방법이떠오르지않는다
3,개금밀면,81.82%,블루리본(2023),True,블루리본(2025),True,가야밀면과함께50년이넘는역사를자랑하는곳.밀가루에고구마전,가야밀면과함께60년의역사를자랑하는곳.밀가루에고구마전분을
4,거대갈비,100.00%,부산의 맛(2024),True,부산의 맛(2025),True,최상등급의고기와최고의원재료만을사용한다는한우숯불구이전문점,최상등급의고기와최고의원재료만을사용한다는한우숯불구이전문점


In [13]:
import re
import pandas as pd
from difflib import SequenceMatcher

# 1. 출처에서 연도 추출 함수 (예: '부산의 맛(2025)' -> 2025)
def extract_year(text):
    year = re.search(r'(\d{4})', str(text))
    return int(year.group(1)) if year else 0  # 연도가 없으면 가장 낮은 0 반환

# 2. 유사도 측정 함수
def similarity_ratio(a, b):
    return SequenceMatcher(None, str(a), str(b)).ratio()

# 3. 최신 데이터만 남기는 필터링 함수
def keep_latest_similar(group):
    # 그룹 내에서 삭제할 인덱스를 저장
    to_drop = set()
    indices = group.index.tolist()
    contents = group['설명_공백제거'].tolist()
    years = group['출처'].apply(extract_year).tolist()
    
    for i in range(len(indices)):
        if indices[i] in to_drop:
            continue
            
        for j in range(i + 1, len(indices)):
            if indices[j] in to_drop:
                continue
                
            # 80% 이상 유사할 경우
            if similarity_ratio(contents[i], contents[j]) >= 0.8:
                # 연도 비교 후 오래된 데이터의 인덱스를 삭제 목록에 추가
                if years[i] >= years[j]:
                    to_drop.add(indices[j])
                else:
                    to_drop.add(indices[i])
                    break # i가 삭제되면 다음 j 비교 무의미
                    
    return group.drop(to_drop)

# 4. 실행: 장소명별로 그룹화하여 처리
df_latest = df.groupby('장소명_공백제거', group_keys=False).apply(keep_latest_similar)

# 결과 확인
df_latest = df_latest.reset_index(drop=True)
print(f"정리 전: {len(df)}행 -> 정리 후: {len(df_latest)}행")

정리 전: 983행 -> 정리 후: 851행


/tmp/ipykernel_334480/3109556089.py:42: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_latest = df.groupby('장소명_공백제거', group_keys=False).apply(keep_latest_similar)


In [14]:
# 1. 전체 데이터에서 '장소명_공백제거'가 겹치는 모든 행 추출
# keep=False: 중복된 쌍을 모두 보여줌 (원본+복사본)
all_duplicates = df_latest[df_latest.duplicated(subset=['장소명_공백제거'], keep=False)]

# 2. 장소명 순으로 정렬하여 비교하기 좋게 배치
all_duplicates_sorted = all_duplicates.sort_values(by='장소명_공백제거')

# 3. 주요 정보 확인 (구분, 장소명, 출처, 설명 등)
# 데이터가 많을 수 있으니 상위 20개만 먼저 확인
print(f"전체 중복 발견 건수: {len(all_duplicates_sorted)}행")
all_duplicates_sorted[['구분', '장소명', '출처', '설명']].head(50)

전체 중복 발견 건수: 251행


,구분,장소명,출처,설명
1,음식점,18번완당집,택슐랭(2024),얇은 만두피로 빚어낸 맛있는 완당을 맛볼 수 있는 76년 전통을 가진 완당 전문점이다.
2,음식점,18번완당집,블루리본(2025),"완당은 중국식 만두의 일종으로, 물만두와 비슷하다. 아주 얇은 피에 채소와 고기를 ..."
5,관광지,40계단,택슐랭(2025),"6.25 피난 시절 부산 중구에 모인 피난민들의 삶과 애환을 상징하는 장소로, 가족..."
6,관광지,40계단,Theme(2025),"복병산 주택가와 해안가 매립지를 연결하는 통로로, 6.25 전쟁 때 헤어진 가족의 ..."
7,음식점,60년전통할매국밥,블루리본(2023),"부추를 넣어 먹는 부산식 돼지국밥이 유명하다. 국물 맛이 깔끔하며, 국밥에 부추를 ..."
8,음식점,60년 전통 할매국밥,부산의 맛(2025),60년 전통의 돼지국밥 전문점으로 식사 시간에는 30분 이상을 기다려야 하는 유명한...
9,음식점,60년전통할매국밥,블루리본(2025),부산의 유명한 돼지국밥 노포 중 하나. 부산의 돼지국밥은 뽀얀 국물을 내는 곳과 맑...
17,문화예술,F1963,Theme(2025),세계 최대 특수 선재 회사인 고려제강의 수영 공장이 문화 공간으로 탈바꿈했다. 이곳...
18,문화예술,F1963,부산한골목-망미골목,‘나만 알고 싶은 공간’이었지만 너무 유명해져 조금은 아쉬운 복합문화공간 F1963...
27,관광지,감천문화마을,사하구 관광안내(2024),"감천문화마을은 한국전쟁 당시 피난민들이 모여 살기 시작하면서 형성된 계단식 마을로,..."


## 데이터 저장

In [15]:
df_latest['구분'].value_counts()

구분
음식점     417
관광지     169
체험       87
카페       81
문화예술     44
쇼핑       31
숙박업체     18
축제        4
Name: count, dtype: int64

In [16]:
# 1. 유지하고 싶은 컬럼 리스트 정의 (실제 데이터의 컬럼명에 맞춰 수정하세요)
selected_cols = [ '장소명', '구분', '설명', '메뉴', '영업시간', '주소', '연락처', '출처'] 

# 2. 해당 컬럼만 추출
df_final_save = df_latest[selected_cols]

# 3. JSON 파일로 저장 
# orient='records': 행 단위의 리스트 형식으로 저장
# force_ascii=False: 한글이 깨지지 않도록 설정
df_final_save.to_json('busan_rag_data.json', orient='records', force_ascii=False, indent=4)

print(f"저장 완료! 총 {len(df_final_save)}개의 데이터가 'busan_rag_data.json'으로 저장되었습니다.")

저장 완료! 총 851개의 데이터가 'busan_rag_data.json'으로 저장되었습니다.
